<a href="https://colab.research.google.com/github/jayden14141/volatility-forecast/blob/main/notebooks/01_fetch_raw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 01 — fetch raw prices

Downloads adjusted closes for QQQ (target) and 8 macro series (VIX, 10y and 3m
yields, HYG, LQD, TLT, GLD) from Yahoo Finance, 2011-01-01 to 2026-05-29, and
writes `data/raw/raw_close.csv`. The end date is fixed; note that Yahoo
back-adjusts history for dividends paid after the fetch date, so a re-download
can differ from an earlier snapshot in late decimal places.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os

# Environment bootstrap: Colab (mount Drive) or a local checkout.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/volatility-forecast"
    if not os.path.isdir(PROJECT_DIR):
        PROJECT_DIR = "/content/volatility-forecast"
        print("Drive not found — falling back to /content")
except ModuleNotFoundError:
    PROJECT_DIR = os.path.abspath(os.getcwd())   # find the repo root locally
    while not os.path.isdir(os.path.join(PROJECT_DIR, "src")):
        parent = os.path.dirname(PROJECT_DIR)
        if parent == PROJECT_DIR:
            raise FileNotFoundError("repo root with src/ not found")
        PROJECT_DIR = parent

RAW_DIR = os.path.join(PROJECT_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)
print("RAW_DIR =", RAW_DIR)

# ticker : QQQ(target) + 8 macro X
TICKERS = ["QQQ", "^VIX", "^TNX", "^IRX", "HYG", "LQD", "TLT", "GLD"]
START, END = "2011-01-01", "2026-05-29"

# download — auto_adjust=True (price assets→adjusted close; indices unaffected)
raw = yf.download(TICKERS, start=START, end=END,
                  auto_adjust=True, group_by="ticker", progress=False)

# extract Close per ticker → outer join (NO dropna, NO fill — raw stays raw)
close = pd.concat({t: raw[t]["Close"] for t in TICKERS}, axis=1)
close.index.name = "Date"
print("shape:", close.shape)

audit = pd.DataFrame({
    "first_valid": close.apply(lambda s: s.first_valid_index()),
    "last_valid":  close.apply(lambda s: s.last_valid_index()),
    "n_nan":       close.isna().sum(),
    "n_rows":      len(close),
})
print(audit)

out_path = os.path.join(RAW_DIR, "raw_close.csv")
close.to_csv(out_path)
print("saved →", out_path)